In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -U transformers
!pip install -q --upgrade accelerate
!pip install -q faiss-cpu
!pip install -q open_clip_torch
!pip install -q evaluate
!pip install -q rouge_score
!pip install -q bert_score
!pip install -q datasets
!pip install -q scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 91.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.2 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 11.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 82.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 24.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.1 MB/s et

In [3]:
import os
import torch
import time
import evaluate
import numpy as np
import gc
import random
import urllib.request
import zipfile
import transformers
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from kaggle_secrets import UserSecretsClient
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset
from PIL import Image, ImageFile
from transformers import AutoModel, AutoTokenizer

# --- 1. CẤU HÌNH & VÁ LỖI KIẾN TRÚC ---
transformers.PreTrainedModel.all_tied_weights_keys = {} # Vá lỗi tied_weights cho transformers 5.0+
transformers.logging.set_verbosity_error()
ImageFile.LOAD_TRUNCATED_IMAGES = True

try:
    userSecrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = userSecrets.get_secret("hf_token") 
except Exception as e:
    print(f"⚠️ Lỗi nạp token: {e}")

imgDir = "/kaggle/working/SLAKE_imgs"

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# --- 2. QUẢN LÝ ẢNH SLAKE TRÊN KAGGLE ---
def ensureImagesExist():
    if not os.path.exists(imgDir) or len(os.listdir(imgDir)) == 0:
        print("⬇️ Đang tải file ảnh gốc (imgs.zip) của SLAKE về /kaggle/working/...")
        urllib.request.urlretrieve("https://huggingface.co/datasets/BoKelvin/SLAKE/resolve/main/imgs.zip", "imgs.zip")
        print("📦 Đang giải nén ảnh...")
        with zipfile.ZipFile("imgs.zip", 'r') as zipRef:
            zipRef.extractall(imgDir)
        os.remove("imgs.zip")
        print("✅ Đã chuẩn bị xong kho ảnh SLAKE cục bộ!")

def getImagePath(imgName):
    if not imgName: return None
    path1 = os.path.join(imgDir, imgName)
    path2 = os.path.join(imgDir, "imgs", imgName)
    if os.path.exists(path1): return path1
    if os.path.exists(path2): return path2
    return None

# --- 3. MÔ HÌNH SUY LUẬN INTERNVL NON-RAG ---
class InternVL8bNonRagGenerator:
    def __init__(self):
        torch.cuda.empty_cache()
        gc.collect()
        print("\n[Hạ tầng] Đang khởi tạo mô hình InternVL3.5-8B (Non-RAG)...")
        
        modelId = "OpenGVLab/InternVL3_5-8B-Instruct"
        self.model = AutoModel.from_pretrained(
            modelId,
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
            device_map="auto"
        ).eval()
        self.tokenizer = AutoTokenizer.from_pretrained(modelId, trust_remote_code=True)
        print(f"✅ Đã nạp thành công mô hình {modelId}!")

    def processImage(self, pilImg, targetSize=448):
        if pilImg.mode != "RGB": 
            pilImg = pilImg.convert("RGB")
        imageTransform = T.Compose([
            T.Resize((targetSize, targetSize), interpolation=InterpolationMode.BICUBIC),
            T.ToTensor(),
            T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
        ])
        return imageTransform(pilImg).unsqueeze(0).to(torch.bfloat16).cuda()

    def generate(self, testImg, testQuest):
        pixelValues = self.processImage(testImg)
        
        # Prompt đơn giản cho Zero-shot
        promptText = f"<image>\n{testQuest}"
        
        generationConfig = dict(max_new_tokens=15, do_sample=False)
        
        with torch.inference_mode():
            response = self.model.chat(
                tokenizer=self.tokenizer,
                pixel_values=pixelValues,
                question=promptText,
                generation_config=generationConfig
            )
        return response.strip()

# --- 4. HÀM THỰC THI CHÍNH ---
def runInternVL8bNonRagBenchmark(sampleSize=500):
    print("1. Khởi tạo và kiểm tra dữ liệu hình ảnh...")
    ensureImagesExist()
    
    print("2. Khởi tạo các thư viện đo lường (Metrics)...")
    bleuMetric = evaluate.load("bleu")
    rougeMetric = evaluate.load("rouge")
    bertscoreMetric = evaluate.load("bertscore")
    
    print("3. Tải Generator...")
    slm = InternVL8bNonRagGenerator()
    
    rawTestDS = load_dataset("BoKelvin/SLAKE", split="test")
    englishTestDS = [item for item in rawTestDS if item.get('q_lang') != 'zh']
    
    totalIndices = list(range(len(englishTestDS)))
    sampledIndices = random.sample(totalIndices, min(sampleSize, len(englishTestDS)))
    
    results = {"closed": {"ref": [], "pred": []}, "open": {"ref": [], "pred": []}}
    latencyList = []
    skippedCount = 0
    
    yesKeywords = ['yes', 'correct', 'true', 'positive', 'present', 'visible']
    noKeywords = ['no', 'incorrect', 'false', 'negative', 'absent', 'normal', 'none']
    
    print(f"\n🚀 BẮT ĐẦU CHẠY LIGHT-MVQA (INTERNVL3.5-8B | NON-RAG) TRÊN {len(sampledIndices)} MẪU SLAKE...")
    print("-" * 60)
    
    if torch.cuda.is_available(): 
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        
    startTotalTime = time.time()

    for progressCount, idx in enumerate(sampledIndices):
        item = englishTestDS[idx]
        imgName = item.get('img_name')
        imgPath = getImagePath(imgName)
        
        if not imgPath: 
            skippedCount += 1
            continue
            
        startTimeSample = time.time()
        
        try:
            rawImg = Image.open(imgPath).convert("RGB")
            
            # Suy luận trực tiếp (Zero-shot)
            aiAnswerRaw = slm.generate(rawImg, str(item['question']))
        except Exception as e:
            print(f"⚠️ Lỗi ngầm tại mẫu {idx}: {e}")
            skippedCount += 1
            continue
            
        latencyList.append(time.time() - startTimeSample)
        
        aiAnswer = aiAnswerRaw.lower().strip()
        groundTruth = str(item['answer']).lower().strip()
        
        if groundTruth in ['yes', 'no']:
            results["closed"]["ref"].append(groundTruth)
            hasYes = any(word in aiAnswer for word in yesKeywords)
            hasNo = any(word in aiAnswer for word in noKeywords)
            
            if hasYes and not hasNo:
                predClean = 'yes'
            elif hasNo and not hasYes:
                predClean = 'no'
            else:
                firstYesPos = aiAnswer.find('yes') if 'yes' in aiAnswer else 9999
                firstNoPos = aiAnswer.find('no') if 'no' in aiAnswer else 9999
                predClean = 'yes' if firstYesPos < firstNoPos else ('no' if firstNoPos < firstYesPos else 'unknown')
            results["closed"]["pred"].append(predClean)
        else:
            results["open"]["ref"].append(groundTruth)
            results["open"]["pred"].append(aiAnswer)
            
        if (progressCount + 1) % 20 == 0: 
            print(f"Tiến độ: {progressCount + 1}/{len(sampledIndices)} mẫu...")

        del rawImg, aiAnswerRaw
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    endTotalTime = time.time()
    
    print("\n" + "="*60 + "\n📊 BÁO CÁO THỰC NGHIỆM - LIGHT-MVQA (INTERNVL3.5-8B ON SLAKE | NON-RAG)\n" + "="*60)
    print(f"📌 Đã quét: {len(sampledIndices)} | Bỏ qua do lỗi: {skippedCount}")
    
    if results["closed"]["ref"]:
        accScore = accuracy_score(results["closed"]["ref"], results["closed"]["pred"])
        f1Weighted = f1_score(results["closed"]["ref"], results["closed"]["pred"], average='weighted', zero_division=0)
        print(f"\n[1. Closed-Ended Classification (Total: {len(results['closed']['ref'])} samples)]")
        print(f"  - Accuracy : {accScore:.4f}")
        print(f"  - F1-Score : {f1Weighted:.4f}")
    
    if results["open"]["ref"]:
        print(f"\n[2. Open-Ended Text Generation (Total: {len(results['open']['ref'])} samples)]")
        predsList = results["open"]["pred"]
        refsList = [[r] for r in results["open"]["ref"]]
        
        bleu1 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=1, smooth=True)['bleu']
        bleu2 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=2, smooth=True)['bleu']
        bleu3 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=3, smooth=True)['bleu']
        rougeL = rougeMetric.compute(predictions=predsList, references=results["open"]["ref"])['rougeL']
        bertResults = bertscoreMetric.compute(predictions=predsList, references=results["open"]["ref"], lang="en")
        
        print(f"  - BLEU-1   : {bleu1:.4f}")
        print(f"  - BLEU-2   : {bleu2:.4f}")
        print(f"  - BLEU-3   : {bleu3:.4f}")
        print(f"  - ROUGE-L  : {rougeL:.4f}")
        print(f"  - BERTScore: {np.mean(bertResults['f1']):.4f}")

    totalDuration = endTotalTime - startTotalTime
    if latencyList:
        print("\n[3. Computational Efficiency Measurements]")
        print(f"  - Inference Latency (Độ trễ trung bình) : {np.mean(latencyList):.2f} s/mẫu")
        print(f"  - Throughput (Tốc độ xử lý)             : {len(latencyList) / totalDuration:.2f} mẫu/giây")
    
    if torch.cuda.is_available():
        print("\n[4. Multi-GPU Memory Profile]")
        for gpuId in range(torch.cuda.device_count()):
            vramMB = torch.cuda.max_memory_allocated(device=gpuId) / (1024 ** 2)
            print(f"  - Peak Memory Usage on GPU {gpuId}        : {vramMB:.2f} MB")
    print("="*60)
    
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

if __name__ == "__main__":
    runInternVL8bNonRagBenchmark(sampleSize=500)

⚠️ Lỗi nạp token: Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 127732856 and label hf_token.'], 'error': {'code': 5}, 'wasSuccessful': False}.
1. Khởi tạo và kiểm tra dữ liệu hình ảnh...
⬇️ Đang tải file ảnh gốc (imgs.zip) của SLAKE về /kaggle/working/...
📦 Đang giải nén ảnh...
✅ Đã chuẩn bị xong kho ảnh SLAKE cục bộ!
2. Khởi tạo các thư viện đo lường (Metrics)...


3. Tải Generator...

[Hạ tầng] Đang khởi tạo mô hình InternVL3.5-8B (Non-RAG)...


config.json: 0.00B [00:00, ?B/s]

configuration_internvl_chat.py: 0.00B [00:00, ?B/s]

configuration_intern_vit.py: 0.00B [00:00, ?B/s]

modeling_internvl_chat.py: 0.00B [00:00, ?B/s]

... (25 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

  - BLEU-1   : 0.1327
  - BLEU-2   : 0.0470
  - BLEU-3   : 0.0119
  - ROUGE-L  : 0.4323
  - BERTScore: 0.9151

[3. Computational Efficiency Measurements]
  - Inference Latency (Độ trễ trung bình) : 3.05 s/mẫu
  - Throughput (Tốc độ xử lý)             : 0.30 mẫu/giây

[4. Multi-GPU Memory Profile]
  - Peak Memory Usage on GPU 0        : 8738.78 MB
  - Peak Memory Usage on GPU 1        : 8693.82 MB
